## LLM as a judge

LLMs are not only used for conversational AI but also to evaluate the conversations. For example, Bavareso et al (2025) tested 11 LLMs against JUDGE-BENCH, a collection of 20 NLLP datasets with human annotations. LLMs are not specififically designed or trained to judge conversations but have some understanding of conversations and should be able to reflect on it. 

In this notebook, we will prompt an LLM to score agent responses for coherence and also to explain how it came to this score. We will use a bigger online LLM through the [ollama platform](https://ollama.com). Obviously, any other platform (OpenAI, Google) or local models can be used for the same purpose. 

We will load the conversations from the EMISSOR representations of conversations by loading pairs of user input and agent responses.

## Reference
Anna Bavaresco, Raffaella Bernardi, Leonardo Bertolazzi, Desmond Elliott, Raquel Fernández, Albert Gatt, Esam Ghaleb, Mario Giulianelli, Michael Hanna, Alexander Koller, Andre Martins, Philipp Mondorf, Vera Neplenbroek, Sandro Pezzelle, Barbara Plank, David Schlangen, Alessandro Suglia, Aditya K Surikuchi, Ece Takmaz, and Alberto Testoni. 2025. LLMs instead of Human Judges? A Large Scale Empirical Study across 20 NLP Evaluation Tasks. In Proceedings of the 63rd Annual Meeting of the Association for Computational Linguistics (Volume 2: Short Papers), pages 238–255, Vienna, Austria. Association for Computational Linguistics.

## Prerequisites
This notebook requires packages installed for running an ```ollama``` client and loading the EMISSOR scenarios. We also need ```pandas``` for saving the results in a spreadsheet.
These packages can be installed through ```pip```:

```
!pip install ollama
!pip install emissor
!pip install pandas
```

In [4]:
import os
import re
from ollama import Client

The [ollama cloud service](https://ollama.com/cloud) requires an API key that can be obtained from the website: [https://docs.ollama.com/cloud#authentication](https://docs.ollama.com/cloud#authentication). Usage is still free but currently limited by hourly and weekly usage.

You need to obtain you own KEY for running this notebook.

## Defining an ollama cloud client and the model

In [5]:
# OpenAI API Key
path = "../../ollama-cloud-key.txt"
api_key = "THIS SHOULD BE YOUR OLLAMA CLOUD API KEY"
with open(path) as f:
    api_key = f.read()

ollama_client = Client(host="https://ollama.com", headers={'Authorization': 'Bearer ' + api_key})

model = "gpt-oss:120b"

## Prompting for coherence

The next two functions define the prompt for evaluating the coherence of the agent responses to the user. The prompt is adapted to take into account that the Leolani agent is proactive in driving the conversation and can ask questions to the user as well.

In [6]:
### differentiates between the role of the user and the assistant
def get_judge_prompt():
    instruction = """
                    ### Role Assignment
                    You are a Coherence Evaluation Judge.
                    Your job is to evaluate how coherent the **assistant’s response** is 
                    with respect to the **user’s input**.
                    The **assistant’s response** may be a follow up question to a 
                    statement from the **user**,  a related statement by the **agent** or the answer 
                    from the **agent** to a question of the **user**.
                    
                    ### Task Definition
                    You must:
                    1. Assign a **coherence score** from **0.0 to 1.0**
                    2. Provide a **short explanation** (maximum 2 sentences)
                    
                    ### Output Format (STRICT)
                    Return ONLY:
                    
                    <JSON>
                    {
                      "coherence_score": float between 0.0 and 1.0,
                      "explanation": "brief rationale"
                    }
                    </JSON>
                    """

    system_prompt = {
        "role": "system",
        "content": instruction
    }
    return system_prompt

# treate the user and assistant as equal interlocutors
def get_judge_prompt_interlocutor():
    instruction = """
                    ### Role Assignment
                    You are a Coherence Evaluation Judge.
                    You will receive a pair of utterances from a **user** and an **assistant** as 
                    interlocutors or the other way around.
                    Your job is to evaluate how coherent an **interlocutor’s response** is with 
                    respect to the previous input of the other **interlocutor**.
                    The **interlocutor’s response** can be a follow up question to a previous statement,  
                    the **interlocutor’s response** can be a related statement 
                    or the **interlocutor’s response** can be an answer to a question of the **user**.
                    Always consider the coherence of the last utterance given the first 
                    utterance in a pair of utterances.
                    
                    ### Task Definition
                    You must:
                    1. Assign a **coherence score** from **0.0 to 1.0**
                    2. Provide a **short explanation** (maximum 2 sentences)
                    
                    ### Output Format (STRICT)
                    Return ONLY:
                    
                    <JSON>
                    {
                      "coherence_score": float between 0.0 and 1.0,
                      "explanation": "brief rationale"
                    }
                    </JSON>
                    """

    system_prompt = {
        "role": "system",
        "content": instruction
    }
    return system_prompt
    
def query_qwen_as_a_judge_ollama(messages, ollama_client, model):
    #system_prompt = get_judge_prompt()
    system_prompt = get_judge_prompt_interlocutor()
    messages_judge = [system_prompt] + messages
    response = ""
    for part in ollama_client.chat(model=model, messages=messages_judge, stream=True, format="json"):
        response += part['message']['content']
    # Look for patterns like "coherence_score": 0.9 or similar
    #     </analysis<|message|>The user said: "Lucy drinks wine". The assistant gave a garbled nonsense response. That's incoherent. Score low, maybe 0.0 or 0.1. Provide explanation.{
    #   "coherence_score": 0.0,
    #   "explanation": "The assistant's output is nonsensical and does not address the simple statement about Lucy drinking wine."
    #    }
    try:
        score_pattern = r'"coherence_score":\s*([\d.]+)'
        explanation_pattern = r'"explanation":\s*"([^"]+)"'

        score_match = re.search(score_pattern, response)
        explanation_match = re.search(explanation_pattern, response)

        if score_match and explanation_match:
            coherence_score = float(score_match.group(1))
            explanation = explanation_match.group(1)
            return {"coherence_score": coherence_score, "explanation": explanation}
        else:
            return {"coherence_score": None, "explanation": "Could not extract score and explanation from response"}
    except Exception as e:
        return {"coherence_score": None, "explanation": f"Error processing response: {str(e)}"}

## Obtaining the conversations from EMISSOR

The next two functions get the text signals from a conversation captured in an EMISSOR scenario and get the speaker of a text signal that represents a turn.

In [7]:
from emissor.persistence import ScenarioStorage
from emissor.representation.scenario import Modality
from emissor.representation.scenario import Signal, TextSignal
import emissor_util as util

In [8]:
EMISSOR="../emissor"
SCENARIO="b387db06-934e-405b-9d4e-f7e5c27b440a"
SCENARIO="2ad0c8b9-4e62-4f11-809c-57bf76487f95"


## Scoring conversational pairs for coherence

The next code shows how the conversation from EMISSOR is processed as pairs of user input and agent responses and each of these is passed on the to LLM for assessing the coherence. It extracts a score for each pair and averages the score over all pairs.

In [ ]:
text_signals = util.get_text_signals_from_a_scenario(EMISSOR, SCENARIO)
coherence_results = []
previous_turn = {"role": "user", "content": ""}
for text_signal in text_signals:
    speaker = util.get_speaker_from_text_signal(text_signal) # SPEAKER or agent
    role = "assistant"
    if not speaker=="LEOLANI":
        role = "user"
    turn = {"role":role, "content": text_signal.text}
    pair = [previous_turn, turn]
    coherence = query_qwen_as_a_judge_ollama(pair, ollama_client, model)
    coherence_score = 0
    explanation = ""
    if "coherence_score" in coherence:
        if not coherence["coherence_score"]==None:
            coherence_score = coherence["coherence_score"]
            explanation = coherence["explanation"]
    print(previous_turn, turn)
    print(coherence)
    row = {"Turn": text_signal.id, "Speaker": speaker, "Response": text_signal.text, "llm_coherence": coherence_score, "llm_explanation": explanation}
    coherence_results.append(row)
    previous_turn = turn


{'role': 'user', 'content': ''} {'role': 'assistant', 'content': "What's up? Do you want to talk to me John?"}
{'coherence_score': 0.0, 'explanation': "The assistant's reply is unrelated and nonsensical given the user's empty input."}
{'role': 'assistant', 'content': "What's up? Do you want to talk to me John?"} {'role': 'user', 'content': 'Yes'}
{'coherence_score': 0.8, 'explanation': 'The user’s “Yes” appropriately answers the assistant’s question about wanting to talk, showing clear relevance, though it doesn’t address the separate “What’s up?” query.'}
{'role': 'user', 'content': 'Yes'} {'role': 'assistant', 'content': 'You have my confidence.'}
{'coherence_score': 0.0, 'explanation': "The assistant's reply is unrelated gibberish and does not address the simple affirmation \\"}
{'role': 'assistant', 'content': 'You have my confidence.'} {'role': 'user', 'content': 'I am John from Oxford'}
{'coherence_score': 0.3, 'explanation': "The assistant's reply does not meaningfully address t

In [ ]:
speaker_coherence = []
agent_coherence = []
for result in coherence_results:
    print(result["Turn"], ":", result["llm_explanation"])
    if result["Speaker"]=="LEOLANI":
        agent_coherence.append(result["llm_coherence"])
    else:
        speaker_coherence.append(result["llm_coherence"])

average_speaker_coherence = sum(speaker_coherence)/len(speaker_coherence)
average_agent_coherence = sum(agent_coherence)/len(agent_coherence)

print('average_speaker_coherence',average_speaker_coherence)
print(speaker_coherence)
print('average_agent_coherence',average_agent_coherence)
print(agent_coherence)

Running the code again likely give you different results. It is wise to run the code several times and average over the different runs.

## Saving the coherence score to a CSV file

We save the conversation with the coherence score to a CSV file to compare it against other evaluations and human scores.

In [8]:
import pandas as pd

In [9]:
df = pd.DataFrame(coherence_results)

evaluation_folder = os.path.join(EMISSOR, SCENARIO, 'evaluation')
if not os.path.exists(evaluation_folder):
    os.mkdir(evaluation_folder)
file_name =  SCENARIO+"_llm_judge_evaluation.csv"
file = os.path.join(evaluation_folder, file_name)
df.to_csv(file, sep=";", index=False)

## End of notebook